In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)

# Load the data
stock_data = pd.read_csv('NSE_data_all_stocks_2026_upto_jun.csv')
sector_data = pd.read_csv('NSE_data_stock_market_sectors_2026.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'NSE_data_stock_market_sectors_2026.csv'

In [ ]:
print("Stock Data Shape:", stock_data.shape)
print("Sector Data Shape:", sector_data.shape)
print("\nStock Data Columns:", stock_data.columns.tolist())
print("\nFirst 5 rows of Stock Data:")
print(stock_data.head())
print("\nSector Data Sample:")
print(sector_data.head())

Stock Data Shape: (9280, 13)
Sector Data Shape: (77, 3)

Stock Data Columns: ['Date', 'Code', 'Name', '12m Low', '12m High', 'Day Low', 'Day High', 'Day Price', 'Previous', 'Change', 'Change%', 'Volume', 'Adjusted Price']

First 5 rows of Stock Data:
       Date  Code                     Name  12m Low  12m High  Day Low  \
0  2-Jan-26  EGAD              Eaagads Ltd     11.0     35.25     18.5   
1  2-Jan-26  KUKZ               Kakuzi Plc    365.0    440.00    390.0   
2  2-Jan-26  KAPC  Kapchorua Tea Kenya Plc    198.0    424.25    230.0   
3  2-Jan-26  LIMT           Limuru Tea Plc    295.0    555.00    460.0   
4  2-Jan-26  SASN               Sasini Plc     13.3     33.90     17.9   

   Day High  Day Price Previous Change Change% Volume Adjusted Price  
0     20.50      20.30     20.5   -0.2  -0.98%   1136              -  
1    405.00     402.00      402      -       -     84              -  
2    235.75     233.00    231.5    1.5   0.65%    699              -  
3    460.00     460.

In [ ]:
# Clean Change columns (missing = 0)
stock_data['Change'] = pd.to_numeric(stock_data['Change'], errors='coerce').fillna(0)
stock_data['Change%'] = pd.to_numeric(stock_data['Change%'], errors='coerce').fillna(0)

# Convert Date
stock_data['Date'] = pd.to_datetime(stock_data['Date'], format='%d-%b-%y')

# Convert numeric columns
price_cols = ['12m Low', '12m High', 'Day Low', 'Day High', 'Day Price', 'Previous', 'Volume']
for col in price_cols:
    stock_data[col] = pd.to_numeric(stock_data[col], errors='coerce')

# Remove indices (^)
stock_data = stock_data[~stock_data['Code'].str.startswith('^', na=False)]

# Merge sectors
stock_data = stock_data.merge(sector_data, left_on='Code', right_on='Stock_code', how='left')

# Remove inactive stocks (Volume sum = 0)
inactive_stocks = stock_data.groupby('Code')['Volume'].sum()[lambda x: x == 0].index
stock_data = stock_data[~stock_data['Code'].isin(inactive_stocks)]

# Fill remaining missing values
stock_data['Volume'] = stock_data['Volume'].fillna(0)
stock_data['Previous'] = stock_data['Previous'].fillna(stock_data['Day Price'])
stock_data['Sector'] = stock_data['Sector'].fillna('Unknown')

# Sort data
stock_data = stock_data.sort_values(['Code', 'Date']).reset_index(drop=True)

print(f"Cleaned data: {stock_data.shape}")
print(f"Unique stocks: {stock_data['Code'].nunique()}")
print(f"Unique sectors: {stock_data['Sector'].nunique()}")

Cleaned data: (7233, 16)
Unique stocks: 62
Unique sectors: 14


In [ ]:
# Engineering features
df = stock_data.copy()

# Lag features (past prices)
for lag in [1, 2, 3, 5, 10]:
    df[f'Lag_{lag}'] = df.groupby('Code')['Day Price'].shift(lag)

#  Rolling averages and volatility
for window in [5, 10, 20]:
    df[f'MA_{window}'] = df.groupby('Code')['Day Price'].transform(
        lambda x: x.rolling(window, min_periods=1).mean())
    df[f'Vol_{window}'] = df.groupby('Code')['Day Price'].transform(
        lambda x: x.rolling(window, min_periods=2).std())

# Price ratios
df['Price_to_High'] = df['Day Price'] / df['12m High']
df['Price_to_Low'] = df['Day Price'] / df['12m Low']
df['Daily_Return'] = df.groupby('Code')['Day Price'].pct_change()

# Date features
df['Day_of_Week'] = df['Date'].dt.dayofweek  # 0=Monday
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter

# Volume features
df['Volume_MA_5'] = df.groupby('Code')['Volume'].transform(
    lambda x: x.rolling(5, min_periods=1).mean())
df['Volume_Ratio'] = df['Volume'] / df['Volume_MA_5']

# Target (next day's price)
df['Target'] = df.groupby('Code')['Day Price'].shift(-1)

print(f"Features engineered: {len(df.columns) - len(stock_data.columns)} new columns")

Features engineered: 20 new columns


In [ ]:
# Performing quality checks

# Missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print("Warning: Missing values found:")
    print(missing[missing > 0])
else:
    print("No missing values")

# Data types
print(f"\nData types: {df.dtypes.value_counts().to_dict()}")

# Date range
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

# Stocks per sector
sector_counts = df.groupby('Sector')['Code'].nunique().sort_values(ascending=False)
print("\nStocks per sector:")
print(sector_counts)

Stock_code        8
Stock_name        8
Lag_1            62
Lag_2           124
Lag_3           185
Lag_5           307
Lag_10          608
Vol_5            62
Vol_10           62
Vol_20           62
Daily_Return     62
Volume_Ratio    240
Target           62
dtype: int64

Data types: {dtype('float64'): 26, dtype('O'): 6, dtype('int32'): 3, dtype('<M8[ns]'): 1}
Date range: 2026-01-02 00:00:00 to 2026-06-30 00:00:00

Stocks per sector:
Sector
Commercial and Services          11
Banking                          11
Manufacturing and Allied          8
Agricultural                      6
Insurance                         6
Energy and Petroleum              5
Investment                        4
Construction and Allied           2
Unknown                           2
Exchange Traded Funds             2
Real Estate Investment Trusts     2
Automobiles and Accessories       1
Investment Services               1
Telecommunication                 1
Name: Code, dtype: int64


In [ ]:
# Exporting data

# Full dataset
df.to_csv('NSE_clean_data_with_features.csv', index=False)
print(" Full dataset saved: NSE_clean_data_with_features.csv")

# Sample data (5 stocks for quick testing)
sample_stocks = ['SCOM', 'KCB', 'EQTY', 'ABSA', 'COOP']
sample_df = df[df['Code'].isin(sample_stocks)]
sample_df.to_csv('NSE_sample_data_5stocks.csv', index=False)
print(f" Sample data saved: NSE_sample_data_5stocks.csv ({len(sample_df)} rows)")

# Data summary
summary = {
    'Total_Rows': len(df),
    'Total_Stocks': df['Code'].nunique(),
    'Total_Sectors': df['Sector'].nunique(),
    'Date_Range': f"{df['Date'].min()} to {df['Date'].max()}",
    'Features_Count': len(df.columns),
    'Sectors_List': sector_counts.to_dict()
}
with open('data_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(" Data summary saved: data_summary.json")

# Feature definitions
feature_definitions = {
    'Price Features': {
        'Day Price': 'Current day closing price',
        'Previous': 'Previous day closing price',
        'Day Low': 'Current day lowest price',
        'Day High': 'Current day highest price',
        '12m Low': '52-week low',
        '12m High': '52-week high'
    },
    'Lag Features': {
        'Lag_1': 'Price from 1 day ago',
        'Lag_2': 'Price from 2 days ago',
        'Lag_3': 'Price from 3 days ago',
        'Lag_5': 'Price from 5 days ago',
        'Lag_10': 'Price from 10 days ago'
    },
    'Rolling Features': {
        'MA_5': '5-day moving average',
        'MA_10': '10-day moving average',
        'MA_20': '20-day moving average',
        'Vol_5': '5-day volatility (std)',
        'Vol_10': '10-day volatility (std)',
        'Vol_20': '20-day volatility (std)'
    },
    'Derived Features': {
        'Price_to_High': 'Current price / 52-week high',
        'Price_to_Low': 'Current price / 52-week low',
        'Daily_Return': 'Daily percentage return',
        'Day_of_Week': 'Day of week (0=Monday)',
        'Month': 'Month number (1-12)',
        'Quarter': 'Quarter number (1-4)',
        'Volume_Ratio': 'Volume / 5-day average volume'
    },
    'Target': {
        'Target': 'Next day\'s closing price (what to predict)'
    }
}
with open('feature_definitions.json', 'w') as f:
    json.dump(feature_definitions, f, indent=2)
print(" Feature definitions saved: feature_definitions.json")

 Full dataset saved: NSE_clean_data_with_features.csv
 Sample data saved: NSE_sample_data_5stocks.csv (610 rows)
 Data summary saved: data_summary.json
 Feature definitions saved: feature_definitions.json


In [5]:
#Price distribution visualization
plt.figure(figsize=(6, 4))
plt.hist(clean['Day Price'].dropna(), bins=50, alpha=0.75, color='steelblue')
plt.title('Distribution of Day Prices')
plt.xlabel('Price (KES)')
plt.show()

NameError: name 'clean' is not defined

<Figure size 600x400 with 0 Axes>

In [6]:
#Volume distribution visualization
plt.figure(figsize=(6, 4))
plt.hist(np.log1p(clean['Volume'].dropna()), bins=50, alpha=0.75, color='seagreen')
plt.title('Distribution of Log(Volume)')
plt.xlabel('Log(Volume)')
plt.show()

NameError: name 'clean' is not defined

<Figure size 600x400 with 0 Axes>

In [ ]:
#Return distribution 
# plt.figure(figsize=(6, 4))
plt.hist(clean['Change%'].dropna(), bins=50, alpha=0.75, color='indianred')
plt.title('Distribution of Daily Change (%)')
plt.xlabel('Change %')
plt.show()

NameError: name 'clean' is not defined

<Figure size 600x400 with 0 Axes>

In [8]:
#Price range by Sector
sector_counts = clean.groupby('Sector', observed=True)['Code'].nunique()
top_sectors = sector_counts[sector_counts >= 3].index.tolist()

plt.figure(figsize=(10, 5))
sns.boxplot(data=clean[clean['Sector'].isin(top_sectors)], x='Sector', y='Day Price')
plt.xticks(rotation=45, ha='right')
plt.title('Price Distribution by Sector')
plt.tight_layout()
plt.show()

NameError: name 'clean' is not defined

In [9]:
#Correlation among price and engineered features
numeric_cols = ['Day Price', 'Previous', 'Day Low', 'Day High', 'Volume', 'Change', 'Change%',
                 'Lag_1', 'Lag_2', 'Lag_3', 'MA_5', 'MA_10', 'MA_20', 'Vol_5', 'Vol_20']

plt.figure(figsize=(7, 5))
sns.heatmap(clean[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


NameError: name 'clean' is not defined

<Figure size 700x500 with 0 Axes>

In [10]:
#Trading Activity
top_volume = clean.groupby('Code', observed=True)['Volume'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(7, 4))
top_volume.sort_values().plot(kind='barh', color='teal')
plt.title('Top 10 Stocks by Total Trading Volume (Jan–Jun 2026)')
plt.xlabel('Total Volume')
plt.tight_layout()
plt.show()

NameError: name 'clean' is not defined

In [11]:
#Day of the week volume pattern
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
clean['Day_Name'] = clean['Date'].dt.day_name()
volume_by_day = clean.groupby('Day_Name')['Volume'].mean().reindex(day_order)

plt.figure(figsize=(7, 4))
volume_by_day.plot(kind='bar', color='steelblue')
plt.title('Average Trading Volume by Day of Week')
plt.ylabel('Average Volume')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(volume_by_day)

NameError: name 'clean' is not defined

In [12]:
#Average returns by Sector
sector_returns = (
    clean[clean['Sector'] != 'Uknown']
    .groupby('Sector', observed=True)['Daily_Return']
    .mean().sort_values(ascending=False)
)

plt.figure(figsize=(7, 4))
colors = ['seagreen' if v >= 0 else 'indianred' for v in sector_returns.values]
sector_returns.plot(kind='barh', color=colors)
plt.title('Average Daily Return by Sector')
plt.xlabel('Average Daily Return')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()

NameError: name 'clean' is not defined

In [ ]:
#Price trend for the most traded stocks
top5 = top_volume.head(5).index.tolist()

plt.figure(figsize=(13, 6))
for code_ in top5:
    sub = clean[clean['Code'] == code_].sort_values('Date')
    plt.plot(sub['Date'], sub['Day Price'], label=code_, linewidth=1.5)

plt.title('Price Trend — Top 5 Most-Traded Stocks')
plt.xlabel('Date')
plt.ylabel('Day Price (KES)')
plt.legend()
plt.tight_layout()